# Register the Customer H2O Model

The first notebook proved that the customer native-binary or MOJO bundle and runtime contract are structurally complete. This notebook registers that same folder as an immutable Azure ML model asset. It does not import, score, or change the model.

## Before you run it

- Finish `01_package_and_validate_model.ipynb` successfully.
- Confirm the printed bundle directory is the intended demo or private customer upload. Replacement steps are in `data/h2o/customer_bundle/README.md`.
- If you changed the model, runtime contract, feature schema, or golden files, rerun notebook 01 and use a new immutable model version.
- Confirm the Azure subscription, resource group, and workspace in `workshop/.env`.
- Leave `REGISTER_H2O_MODEL=false` for the first pass. This lets you inspect the model definition without writing to Azure.

We will validate the bundle again, connect to the target workspace, prepare the model definition, and register only when the safety switch is enabled.

**Source:** Adapted from this repository's H2O onboarding notebook and the Azure ML model asset examples.

In [1]:
from pathlib import Path
import os
import sys

from azure.ai.ml import MLClient
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml.entities import Model
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

notebook_file = globals().get("__vsc_ipynb_file__")
search_start = (
    Path(notebook_file).resolve().parent
    if notebook_file
    else Path.cwd().resolve()
)
for candidate in (search_start, *search_start.parents):
    if (candidate / ".env.example").is_file() and (candidate / "outputs").is_dir():
        WORKSHOP_ROOT = candidate
        break
else:
    raise FileNotFoundError("Run this notebook from inside the workshop folder")

load_dotenv(WORKSHOP_ROOT / ".env", override=True)
sys.path.insert(0, str(WORKSHOP_ROOT / "src/h2o"))
from customer_profiles import load_selected_manifest
from validate_bundle import validate_bundle

print(f"Workshop root: {WORKSHOP_ROOT}")

Workshop root: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop


## 1. Recheck the local bundle

Registration should never bypass the intake gate. We read the existing manifest, verify every checksum and format-specific metadata field again, and require packaging validation to be `passed`. Runtime golden validation remains pending when fixtures were provided and `not_provided` otherwise.

In [2]:
BUNDLE_DIR, manifest = load_selected_manifest(WORKSHOP_ROOT)
MODEL_PATH = BUNDLE_DIR / manifest["model_file"]
MODEL_NAME = manifest["model_name"]
REGISTER = (
    os.getenv("REGISTER_H2O_MODEL", "false").lower() in {"1", "true", "yes"}
)

summary = validate_bundle(
    BUNDLE_DIR,
    manifest["runtime_h2o_version"],
    require_packaging_validation=True,
)

display(summary)

{'bundle_dir': '/home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/h2o_customer_bundle',
 'model_name': 'workshop-h2o-customer-binary',
 'model_version': '1',
 'model_format': 'h2o_binary',
 'h2o_version': '3.46.0.12',
 'runtime_h2o_version': '3.46.0.12',
 'mojo_version': None,
 'model_category': None,
 'model_file': 'taxi-fare-gbm',
 'features': ['vendorID',
  'passengerCount',
  'tripDistance',
  'paymentType',
  'pickupHour'],
 'golden_provided': True,
 'golden_validation_enabled': True,
 'golden_required': False,
 'golden_input_file': 'golden_input.csv',
 'golden_expected_file': 'golden_expected.csv',
 'golden_rows': 20,
 'checksums': 'passed',
 'packaging_validation': 'passed',
 'golden_validation': 'pending'}

## 2. Confirm the target workspace

The next cell authenticates with your Azure CLI session and reads the workspace named in `.env`. This is a read-only check. Make sure the printed workspace and resource group are the customer environment you intend to use.

In [ ]:
credential = AzureCliCredential(
    tenant_id=os.getenv("AZURE_TENANT_ID") or None
)
ml_client = MLClient(
    credential,
    os.environ["AZURE_SUBSCRIPTION_ID"],
    os.environ["AZURE_RESOURCE_GROUP"],
    os.environ["AZUREML_WORKSPACE_NAME"],
)

workspace = ml_client.workspaces.get(os.environ["AZUREML_WORKSPACE_NAME"])
print(f"Target workspace: {workspace.name}")
print(f"Resource group: {os.environ['AZURE_RESOURCE_GROUP']}")
print(f"Registration enabled: {REGISTER}")

## 3. Prepare the Azure ML model definition

Azure ML registers the whole bundle directory as a custom model. The tags carry the artifact format, producer version, selected runtime, and validation state into the asset catalog so an operator can check provenance without opening the files.

Review the printed name, version, path, and tags before registration.

In [4]:
model_definition = Model(
    name=MODEL_NAME,
    type=AssetTypes.CUSTOM_MODEL,
    path=str(BUNDLE_DIR),
    description="Packaged customer H2O model bundle",
    tags={
        "workshop": "azureml-h2o-customer",
        "model_format": summary["model_format"],
        "model_h2o_version": summary["h2o_version"],
        "runtime_h2o_version": summary["runtime_h2o_version"],
        "mojo_version": summary["mojo_version"] or "not_applicable",
        "packaging_validation": summary["packaging_validation"],
        "runtime_validation": summary["golden_validation"],
    },
)

print(f"Asset: {model_definition.name} (version assigned on registration)")
print(f"Type: {model_definition.type}")
print(f"Path: {model_definition.path}")
display(model_definition.tags)

Asset: workshop-h2o-customer-binary (version assigned on registration)
Type: custom_model
Path: /home/azureuser/MLOPs-AzureML-workshop-277d1b6/workshop/outputs/h2o_customer_bundle


{'workshop': 'azureml-h2o-customer',
 'model_format': 'h2o_binary',
 'model_h2o_version': '3.46.0.12',
 'runtime_h2o_version': '3.46.0.12',
 'mojo_version': 'not_applicable',
 'packaging_validation': 'passed',
 'runtime_validation': 'pending'}

## 4. Register and verify

This is the only cell that writes to Azure. With the switch off, it prints what is ready and stops. With the switch on, it registers the model and reads it back to confirm that packaging validation and target-runtime metadata were preserved.

When registration is enabled, Azure ML assigns the next asset version. Return `REGISTER_H2O_MODEL` to `false` unless you are intentionally registering another version.

In [5]:
if REGISTER:
    registered_model = ml_client.models.create_or_update(model_definition)
    verified_model = ml_client.models.get(
        MODEL_NAME, version=registered_model.version
    )

    assert verified_model.tags["packaging_validation"] == "passed"
    assert verified_model.tags["runtime_validation"] == summary["golden_validation"]
    print(f"Registered model: {verified_model.name}:{verified_model.version}")
else:
    print(f"Validated model definition: {MODEL_NAME} (version assigned on registration)")
    print("Registration is off. Set REGISTER_H2O_MODEL=true when you are ready.")


Uploading h2o_customer_bundle (0.14 MBs):   0%|      | 0/142313 [00:00<?, ?it/s]


Uploading h2o_customer_bundle (0.14 MBs): 100%|█| 142313/142313 [00:00<00:00, 42

Registered model: workshop-h2o-customer-binary:1


## Expected Result

The packaged customer model is registered as an immutable custom-model version with artifact format, producer, runtime, and validation-state tags.

Next: `03_create_environment.ipynb`.